<div dir="rtl">

# HM_02 — מעבדה: לממש ולמדוד את המנגנון של מודל שפה

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע · **מפגש 2:** איך מודל שפה "חושב" קוד?

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 2 למדנו שרשרת של מושגים: **טקסט ← טוקנים ← embeddings ← התפלגות על הטוקן הבא ← בחירה (טמפרטורה)**, ולמה בסוף השרשרת הזו נוצרות **הזיות**. במעבדה הזו תממשו בעצמכם את הפונקציות שבלב השרשרת, ותשתמשו בהן כדי לבדוק במדידה את הטענות מההרצאה.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | טוקניזציה (BPE) | `word_frequencies`, `count_pairs`, `merge_pair`, `merge_corpus` | 14 דק' |
| ב' | Embeddings | `cosine` | 5 דק' |
| ג' | טמפרטורה | `softmax`, `p_greedy_path` | 10 דק' |
| ד' | הזיות | `audit` | 8 דק' |
| ה' | ניסוי חבילות | `jaccard` | 6 דק' |
| ו' | מסקנה | — | 2 דק' |

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` — **בעצמכם, בלי כלי AI**. המטרה היא להבין את המנגנון, לא לקבל קוד. (בחלק ה' משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל מדידה כותבים תחזית** בתא המיועד לכך. ההבדל בין התחזית לתוצאה הוא חלק מהציון — תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** בתאי ה"תשובה". תשובה טובה מסבירה **במונחי המנגנון** מההרצאה.

## הגשה

בסוף: **Runtime → Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל הפונקציות, ואז **File → Download → Download .ipynb** והגישו את הקובץ.

</div>

In [ ]:
# תא הכנה — הריצו פעם אחת בתחילת העבודה
import math
import random
import inspect
from collections import Counter

import requests

print("✓ מוכן")

<div dir="rtl">

---

# חלק א' — טוקניזציה: לממש BPE בעצמכם (14 דק')

## הרקע — אותה דוגמה מההרצאה

מודל שפה לא קורא אותיות או מילים — הוא קורא **טוקנים**. אוצר הטוקנים $\mathcal{V}$ נבנה לפני אימון המודל, באלגוריתם **BPE**. בחלק הזה תבנו את אותו אוצר מילים שבנינו בהרצאה, על אותו קורפוס:

`hug hug hug hugs bug bugs bug bugs hug hug hugs hugs`

**האלגוריתם, כמו בהרצאה:**

1. סופרים כמה פעמים כל מילה מופיעה בקורפוס — השכיחות $f(w)$.
2. מפרקים כל מילה לאותיות. בהתחלה $\mathcal{V}$ הוא קבוצת האותיות.
3. **צעד:** סופרים כל זוג טוקנים צמודים, כשכל הופעה בתוך מילה $w$ נספרת $f(w)$ פעמים: $\text{count}(a, b) = \sum_{w} f(w) \cdot c_w(a, b)$, כאשר $c_w(a, b)$ הוא מספר הפעמים שהזוג מופיע צמוד בתוך $w$.
4. ממזגים את הזוג השכיח ביותר לטוקן חדש, מוסיפים אותו ל-$\mathcal{V}$, ומחליפים אותו בכל המילים.
5. חוזרים לשלב 3.

**התוצאה שצריכה לצאת לכם** (הטבלה מההרצאה):

| צעד | הזוג השכיח ביותר (ספירה) | טוקן חדש |
|---|---|---|
| 1 | `u · g` (12) | `ug` |
| 2 | `h · ug` (8) | `hug` |
| 3 | `b · ug` (4) | `bug` |
| 4 | `hug · s` (3) | `hugs` |

תממשו ארבע פונקציות קטנות — אחת לכל שלב — ותא נתון יחבר אותן לאלגוריתם המלא.

</div>

<div dir="rtl">

## שלב 1 — השכיחויות $f(w)$

ממשו פונקציה שמקבלת את הקורפוס כמחרוזת ומחזירה מילון: לכל מילה — כמה פעמים היא מופיעה.

**כלים:** `text.split()` מפרק מחרוזת לרשימת מילים לפי רווחים. `Counter(list)` סופר כמה פעמים מופיע כל איבר ברשימה (ו-`dict(...)` הופך את התוצאה למילון רגיל).

</div>

In [ ]:
def word_frequencies(text):
    """Returns a dict word -> how many times it appears in text (words are separated by spaces).
    e.g. word_frequencies("a b a") -> {"a": 2, "b": 1}
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — שלב 1
text = "hug hug hug hugs bug bugs bug bugs hug hug hugs hugs"

def check_a1():
    assert word_frequencies("a b a") == {"a": 2, "b": 1}, "word_frequencies: הדוגמה מה-docstring"
    assert word_frequencies(text) == {"hug": 5, "hugs": 3, "bug": 2, "bugs": 2}, \
        "word_frequencies: השכיחויות של הקורפוס מההרצאה צריכות להיות 5, 3, 2, 2"
    return True

check_a1() and print("✓ שלב 1:", word_frequencies(text))

<div dir="rtl">

## הייצוג — כל מילה כרצף טוקנים (נתון)

כדי למזג טוקנים, כל מילה צריכה להיות **רצף של טוקנים**, לא מחרוזת. נייצג את הקורפוס כמילון: **המפתח** הוא המילה כ-tuple של טוקנים, **הערך** הוא השכיחות שלה. בהתחלה כל טוקן הוא אות אחת. הריצו את התא וראו את המבנה:

</div>

In [ ]:
def to_letters(freqs):
    """Turns {"hugs": 3} into {("h", "u", "g", "s"): 3} — every word becomes a tuple of letters."""
    return {tuple(word): f for word, f in freqs.items()}

corpus = to_letters(word_frequencies(text))
for tokens, f in corpus.items():
    print(f"{f} × {tokens}")

<div dir="rtl">

## שלב 2 — ספירת זוגות $\text{count}(a, b)$

ממשו פונקציה שמקבלת את הקורפוס (בייצוג שלמעלה) ומחזירה, לכל זוג טוקנים צמודים $(a, b)$, את $\text{count}(a, b)$.

**איך:** עוברים על כל מילה (`tokens`) והשכיחות שלה (`f`). בתוך המילה, עוברים על כל המקומות $k$ מ-0 עד `len(tokens) - 2`, ולזוג `(tokens[k], tokens[k + 1])` מוסיפים `f`.

**לדוגמה:** במילה `('h', 'u', 'g', 's')` עם שכיחות 3 יש שלושה זוגות צמודים: `('h','u')`, `('u','g')`, `('g','s')` — כל אחד מקבל $+3$. אחרי שעוברים על כל ארבע המילים, `('u','g')` צריך להגיע ל-$5 + 3 + 2 + 2 = 12$.

**כלי:** `Counter()` הוא מילון שערך ברירת המחדל שלו 0, כך ש-`counts[key] += f` עובד גם למפתח חדש.

</div>

In [ ]:
def count_pairs(corpus):
    """corpus: dict tokens_tuple -> frequency, e.g. {("h", "u", "g"): 5, ...}
    returns: Counter mapping each adjacent pair (a, b) -> count(a, b)
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — שלב 2
def check_a2():
    assert count_pairs({("a", "b"): 1}) == Counter({("a", "b"): 1}), "count_pairs: מילה אחת, זוג אחד"
    assert count_pairs({("a", "b", "a", "b"): 2}) == Counter({("a", "b"): 4, ("b", "a"): 2}), \
        "count_pairs: זוג שמופיע פעמיים באותה מילה נספר פעמיים (כפול השכיחות)"
    assert count_pairs(to_letters(word_frequencies(text))) == \
        Counter({("u", "g"): 12, ("h", "u"): 8, ("g", "s"): 5, ("b", "u"): 4}), \
        "count_pairs: הספירות בצעד 1 של ההרצאה צריכות להיות 12, 8, 5, 4"
    return True

check_a2() and print("✓ שלב 2:", dict(count_pairs(corpus)))

<div dir="rtl">

## שלב 3 — מיזוג זוג בתוך מילה אחת

ממשו פונקציה שמקבלת מילה (tuple של טוקנים) וזוג $(a, b)$, ומחזירה tuple חדש שבו כל הופעה של $a$ ומיד אחריו $b$ הוחלפה בטוקן אחד, $a + b$ (שרשור של שתי המחרוזות).

**איך:** סורקים משמאל לימין עם אינדקס `k`. אם `tokens[k]` ו-`tokens[k + 1]` הם הזוג — מוסיפים לתוצאה את `tokens[k] + tokens[k + 1]` ו**מדלגים שניים** (`k += 2`). אחרת — מוסיפים את `tokens[k]` ומתקדמים אחד.

**לדוגמה:** `merge_pair(('h','u','g','s'), ('u','g'))` ← `('h', 'ug', 's')`. ולמה מדלגים שניים? ב-`('a','a','a')` עם הזוג `('a','a')`: ממזגים את שתי הראשונות ל-`'aa'`, והאות השלישית נשארת לבד — `('aa', 'a')`. האות השנייה כבר "נוצלה" ולא משתתפת במיזוג נוסף.

</div>

In [ ]:
def merge_pair(tokens, pair):
    """tokens: tuple of tokens, e.g. ("h", "u", "g", "s"); pair: (a, b), e.g. ("u", "g")
    returns: a new tuple where every adjacent a, b became the single token a + b,
             e.g. ("h", "ug", "s")
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — שלב 3
def check_a3():
    assert merge_pair(("h", "u", "g", "s"), ("u", "g")) == ("h", "ug", "s"), "merge_pair: הדוגמה מה-docstring"
    assert merge_pair(("h", "ug"), ("h", "ug")) == ("hug",), "merge_pair: מיזוג שהופך את כל המילה לטוקן אחד"
    assert merge_pair(("a", "a", "a"), ("a", "a")) == ("aa", "a"), "merge_pair: אחרי מיזוג מדלגים על שני הטוקנים"
    assert merge_pair(("b", "u", "s", "h"), ("u", "g")) == ("b", "u", "s", "h"), "merge_pair: זוג שלא קיים — לא משנים כלום"
    return True

check_a3() and print("✓ שלב 3")

<div dir="rtl">

## שלב 4 — מיזוג בכל הקורפוס

ממשו פונקציה שמפעילה את `merge_pair` על **כל** מילה בקורפוס, ומחזירה קורפוס חדש עם אותן שכיחויות.

**לדוגמה** (צעד 1 בהרצאה): אחרי מיזוג `('u','g')`, הקורפוס `{('h','u','g'): 5, ('h','u','g','s'): 3, ...}` הופך ל-`{('h','ug'): 5, ('h','ug','s'): 3, ('b','ug'): 2, ('b','ug','s'): 2}`.

</div>

In [ ]:
def merge_corpus(corpus, pair):
    """Applies merge_pair to every word in corpus; frequencies stay the same.
    returns: a new dict tokens_tuple -> frequency
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — שלב 4
def check_a4():
    after = merge_corpus(to_letters(word_frequencies(text)), ("u", "g"))
    assert after == {("h", "ug"): 5, ("h", "ug", "s"): 3, ("b", "ug"): 2, ("b", "ug", "s"): 2}, \
        "merge_corpus: הקורפוס אחרי צעד 1 של ההרצאה"
    return True

check_a4() and print("✓ שלב 4")

<div dir="rtl">

## האלגוריתם המלא (נתון) — השוו להרצאה

התא הבא משתמש בארבע הפונקציות שלכם כדי לבנות את אוצר המילים, ומדפיס כל צעד. **הפלט צריך להיות זהה לטבלה מההרצאה.** (אם יש תיקו בספירה, `max` עם `key=(ספירה, זוג)` בוחר תמיד את אותו זוג, כך שהתוצאה קבועה.)

אחר כך, `tokenize` מפרקת מילה חדשה בעזרת המיזוגים שנלמדו, **לפי הסדר שבו נלמדו** — זה הטוקנייזר $\tau$ מההרצאה.

**תחזית (לפני ההרצה):** לכמה טוקנים יתפרקו `hugs`, `bugs`, `bush` ו-`ממוצע`?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
def train_bpe(text, num_merges):
    """Builds a BPE vocabulary from text and returns the list of merges, in order."""
    corpus = to_letters(word_frequencies(text))
    vocab = sorted({ch for tokens in corpus for ch in tokens})
    print("אוצר מילים התחלתי:", vocab)
    merges = []
    for step in range(1, num_merges + 1):
        pairs = count_pairs(corpus)
        if not pairs:                                   # every word is already one token
            break
        best = max(pairs, key=lambda p: (pairs[p], p))  # the most frequent pair
        merges.append(best)
        corpus = merge_corpus(corpus, best)
        vocab.append(best[0] + best[1])
        print(f"צעד {step}: הזוג השכיח ביותר {best[0]} · {best[1]} ({pairs[best]}) → טוקן חדש '{best[0] + best[1]}'")
    print("אוצר המילים הסופי:", vocab)
    return merges


def tokenize(word, merges):
    """The tokenizer tau: starts from letters and applies the learned merges in order."""
    tokens = tuple(word)
    for pair in merges:
        tokens = merge_pair(tokens, pair)
    return tokens


merges = train_bpe(text, 4)
print()
for w in ["hug", "hugs", "bug", "bugs", "bush", "ממוצע"]:
    t = tokenize(w, merges)
    print(f"τ({w}) = {t}   n = {len(t)}")

<div dir="rtl">

## ניסוי — מה קורה אם משנים את השכיחויות?

**תחזית:** אם נוסיף לקורפוס עוד שתי הופעות של `bugs` (כלומר $f(\texttt{bugs}) = 4$), מה ישתנה בטבלת המיזוגים? לכמה טוקנים יתפרקו עכשיו `hugs` ו-`bugs`?

**התחזית שלי:** _(כתבו כאן)_

בתא הבא, הגדירו `text2` עם שתי ההופעות הנוספות, הריצו `train_bpe(text2, 4)` ופרקו את `hugs` ואת `bugs`.

</div>

In [ ]:
# TODO: הניסוי

<div dir="rtl">

## השוואה למפענח אמיתי

התא הבא משתמש במפענח `o200k_base` — המפענח של מודלים של OpenAI מ-2024. הוא נבנה באותו עיקרון בדיוק, רק על כמות עצומה של טקסט ועם כמאתיים אלף מיזוגים. הפלט מציג כל טוקן מופרד בקו `|`.

</div>

In [ ]:
!pip install -q tiktoken
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
for t in ["calculate_total_price_with_vat",
          "Write a function that computes the average of a list of numbers",
          "כתוב פונקציה שמחשבת ממוצע של רשימת מספרים"]:
    ids = enc.encode(t)
    print(f"{len(ids):3} טוקנים | {' | '.join(enc.decode([i]) for i in ids)}")

<div dir="rtl">

## שאלות ניתוח — חלק א'

**(א)** למה `hugs` הפכה לטוקן אחד ו-`bugs` נשארה שני טוקנים, למרות שהן בנויות באותה צורה? הסבירו בעזרת הספירות בצעד 4.

**תשובה:** _(כתבו כאן)_

**(ב)** מה יצא בניסוי עם $f(\texttt{bugs}) = 4$? האם התחזית שלכם הייתה נכונה? מה זה אומר על הקשר בין שכיחות לבין מספר הטוקנים?

**תשובה:** _(כתבו כאן)_

**(ג) קיבולת.** לפי הפלט של המפענח האמיתי, חשבו את יחס הטוקנים-למילה $\rho = n / n_\text{words}$ בעברית ובאנגלית (ספרו את המילים בכל משפט). בחלון הקשר של $L = 128{,}000$ טוקנים, שמתוכם $8{,}000$ שמורים לתשובה — כמה מילים בעברית נכנסות, וכמה באנגלית? (אפשר לחשב בתא קוד.)

**תשובה:** _(כתבו כאן)_

**(ד)** במפענח האמיתי, `calculate_total_price_with_vat` מתפרק ל-`calculate | _total | _price | _with | _v | at`. איזה חלק בו מתנהג כמו `bugs` אצלכם, ולמה? ולמה `ממוצע` לא התמזגה בכלל במפענח שלכם?

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# תא חישוב לשאלה (ג)

<div dir="rtl">

---

# חלק ב' — Embeddings: לממש דמיון קוסינוס (5 דק')

## הרקע

אחרי הטוקניזציה, כל טוקן ממופה ל**וקטור** — רשימה ארוכה של מספרים, שאפשר לחשוב עליה כ"כתובת" של נקודה במרחב רב-ממדי. המיקומים נלמדים באימון, כך שטוקנים שמופיעים **בהקשרים דומים** מקבלים מיקומים **קרובים**. קרבה במרחב ≈ קרבה במשמעות.

## הנוסחה

מודדים קרבה בעזרת **דמיון קוסינוס** — הקוסינוס של הזווית בין שני הווקטורים:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}, \qquad \mathbf{a} \cdot \mathbf{b} = \sum_{k=1}^{d} a_k b_k, \qquad \lVert \mathbf{a} \rVert = \sqrt{\mathbf{a} \cdot \mathbf{a}} = \sqrt{\sum_{k=1}^{d} a_k^2}$$

כאן $\mathbf{a} = (a_1, \dots, a_d)$ ו-$\mathbf{b} = (b_1, \dots, b_d)$ הם שני וקטורים באורך $d$ (למשל שני embeddings), ו-$a_k$ הוא הרכיב ה-$k$ של $\mathbf{a}$.

- $\mathbf{a} \cdot \mathbf{b}$ — **מכפלה סקלרית**: מכפילים איבר-איבר ומסכמים.
- $\lVert \mathbf{a} \rVert$ — **אורך** (נורמה) של הווקטור.
- התוצאה בין $-1$ ל-$1$: ערך **1** = אותו כיוון בדיוק; **0** = ניצבים (אין קשר); **‎−1** = כיוונים הפוכים.

**דוגמה ידנית:** $\mathbf{a} = (1, 0)$, $\mathbf{b} = (1, 1)$: $\mathbf{a} \cdot \mathbf{b} = 1 \cdot 1 + 0 \cdot 1 = 1$, $\lVert \mathbf{a} \rVert = 1$, $\lVert \mathbf{b} \rVert = \sqrt{2}$, ולכן $\cos = 1/\sqrt{2} \approx 0.707$ (זווית של 45°).

</div>

In [ ]:
def cosine(a, b):
    """Cosine similarity of two equal-length vectors a and b:
    (a · b) / (|a| * |b|), where a · b = sum of a[k]*b[k] and |a| = sqrt(a · a).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — חלק ב' (וקטורי הצעצוע מההרצאה: [חיה?, מחמד?, כלי תחבורה?])
def check_b():
    dog, cat, car = [0.90, 0.80, 0.10], [0.85, 0.75, 0.20], [0.10, 0.20, 0.95]
    assert abs(cosine([1, 0], [1, 1]) - 1 / math.sqrt(2)) < 1e-12, "cosine: הדוגמה הידנית (0.707)"
    assert round(cosine(dog, cat), 3) == 0.996, "cosine(כלב, חתול) צריך להיות 0.996"
    assert round(cosine(dog, car), 3) == 0.293, "cosine(כלב, מכונית) צריך להיות 0.293"
    assert abs(cosine([1, 0], [0, 1])) < 1e-12, "וקטורים ניצבים — דמיון 0"
    assert abs(cosine([2, 2], [1, 1]) - 1) < 1e-12, "אותו כיוון, אורך שונה — דמיון 1"
    return True

check_b() and print("✓ חלק ב' — המימוש עובר את הבדיקות")

<div dir="rtl">

## שאלות ניתוח — חלק ב'

**(א) תחזית ואז חישוב:** מה יחזיר `cosine([1, 2, 3], [-1, -2, -3])`? כתבו תחזית, הריצו בתא שלמטה, והסבירו את משמעות הערך.

**התחזית שלי:** _(כתבו כאן)_

**תשובה:** _(כתבו כאן)_

**(ב)** הבדיקה האחרונה אומרת ש-`[2, 2]` ו-`[1, 1]` "זהים" (דמיון 1), למרות שהמרחק ביניהם אינו 0. למה זו תכונה **רצויה** כשמשווים משמעות?

**תשובה:** _(כתבו כאן)_

**(ג) הקשר להזיות:** נניח שהדמיון בין ה-embedding של `get` לזה של `get_value` גבוה מאוד. איך זה יכול להסביר את ההזיה שתמצאו בחלק ד'?

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# תא לשאלה (א)

<div dir="rtl">

---

# חלק ג' — טמפרטורה: לממש softmax, ולמדוד מה קורה לתשובה שלמה (10 דק')

## הרקע

בכל צעד, המודל נותן **ציון גולמי** (logit) $z_t$ לכל טוקן אפשרי $t$. הציונים הופכים להסתברויות, ונבחר טוקן אחד **בדגימה** מתוך ההתפלגות. הטוקן שנבחר מתווסף לקלט, וחוזרים על התהליך — טוקן אחר טוקן.

## הנוסחה — softmax עם טמפרטורה

$$p(t) = \frac{e^{z_t / T}}{\sum_{t'} e^{z_{t'} / T}}$$

$p(t)$ היא ההסתברות שייבחר הטוקן $t$, ו-$t'$ הוא אינדקס סכימה שרץ על כל הטוקנים המועמדים (כך שהמכנה זהה לכל $t$, וסכום ההסתברויות הוא 1).

- $T$ קטן (למשל 0.2) ← ההבדלים בין הציונים מתעצמים ← כמעט תמיד נבחר הטוקן המוביל.
- $T = 1$ ← ההתפלגות כפי שהמודל חישב.
- $T$ גדול (למשל 1.5) ← ההתפלגות "שטוחה" ← אפשרויות פחות סבירות נבחרות יותר.

## בעיה נומרית — גלישה (overflow)

מספר עשרוני (`float`) במחשב לא יכול להיות גדול מכ-$1.8 \times 10^{308}$. כבר $e^{710}$ גדול מזה, ולכן `math.exp(1000)` זורק `OverflowError`. ציונים כאלה לא נדירים במודלים אמיתיים, ולכן מימוש "ישיר" של הנוסחה **שבור**.

**הרמז לפתרון:** אם מחסירים **אותו קבוע** $c$ מכל הציונים, התוצאה של softmax **לא משתנה** (תוכיחו זאת בשאלה ב'). בחרו $c$ שמבטיח שאף מעריך לא יהיה גדול מ-0. (מה הערך של $e^0$? ומה המעריך הגדול ביותר שיכול להופיע?)

## המודל לתשובה שלמה

תשובה של 200 טוקנים מורכבת מ-190 טוקנים **קלים** (כמעט כפויים — למשל סוגר אחרי פתיחת סוגריים) ו-10 **נקודות החלטה** (כמו בחירת ההמשך של `for i in ...`). נשאל: מה ההסתברות שכל 200 הטוקנים יהיו בדיוק הבחירה המובילה — כלומר שהתשובה תהיה **זהה** לתשובה שמתקבלת ב-$T = 0$?

נסמן ב-$\pi_\text{easy}$ את ההסתברות של הטוקן המוביל בטוקן קל, וב-$\pi_\text{dec}$ את ההסתברות של הטוקן המוביל בנקודת החלטה (שתיהן מחושבות ב-softmax, עם אותה $T$). אם הטוקנים **בלתי-תלויים**, ההסתברות שכולם יהיו המובילים היא מכפלה:

$$P(\text{תשובה זהה}) = \pi_\text{easy}^{\,n_\text{easy}} \cdot \pi_\text{dec}^{\,n_\text{dec}}$$

**הערה:** בשתי רשימות הציונים שנשתמש בהן, הציון הראשון הוא הגבוה ביותר — כך שההסתברות של הטוקן המוביל היא פשוט האיבר הראשון ברשימת ההסתברויות.

</div>

In [ ]:
def softmax(z, T):
    """Turns raw scores z into probabilities with temperature T:
    p(t) = exp(z_t / T) / (sum over all candidates t' of exp(z_t' / T))
    Must not overflow for large scores (e.g. z = [1000, 999]).
    Hint: subtracting the same constant c from every z_t does not change the result.
    """
    # TODO
    raise NotImplementedError


def p_greedy_path(T, z_easy, z_decision, n_easy=190, n_decision=10):
    """Probability that EVERY token of an answer is the top choice, assuming independent tokens:
    n_easy "easy" tokens with scores z_easy and n_decision "decision" tokens with scores z_decision.
    The top choice is the first score in each list.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — חלק ג'
z_decision = [4.0, 2.7, 1.7, 1.2, 0.5]   # נקודת החלטה — הטבלה מההרצאה ("for i in ...")
z_easy = [8.0, 2.0, 1.0]                  # טוקן "קל", כמעט כפוי

def check_c():
    p = softmax(z_decision, 1.0)
    assert abs(sum(p) - 1) < 1e-9, "softmax: סכום ההסתברויות חייב להיות 1"
    assert round(p[0], 3) == 0.683, "softmax ב-T=1: האפשרות המובילה צריכה לקבל 0.683"
    assert round(softmax(z_decision, 0.2)[0], 3) == 0.998, "softmax ב-T=0.2: המובילה צריכה לקבל 0.998"
    assert softmax([1000, 999], 1.0)[0] > 0.7, "softmax: לא יציב נומרית (ראו את הרמז)"
    assert round(p_greedy_path(1.0, z_easy, z_decision), 4) == 0.0116, "p_greedy_path ב-T=1 צריך להיות 0.0116"
    return True

check_c() and print("✓ חלק ג' — המימוש עובר את הבדיקות")

<div dir="rtl">

## מדידה

**תחזית (לפני ההרצה):** ב-$T = 1$, מה הסיכוי (באחוזים) שתשובה של 200 טוקנים תהיה זהה לתשובה המובילה?

**התחזית שלי:** _(כתבו כאן)_

התא הבא עושה שני דברים: (1) דוגם 1,000 פעמים טוקן בודד ומשווה את השכיחות שנמדדה להסתברות התאורטית; (2) מסמלץ 1,000 "תשובות" שלמות ובודק בכמה מהן כל הטוקנים היו המובילים — ומשווה לחישוב האנליטי שלכם.

</div>

In [ ]:
random.seed(0)
for T in [0.2, 0.7, 1.0, 1.5]:
    p = softmax(z_decision, T)
    sample = random.choices(range(len(p)), weights=p, k=1000)
    print(f"T={T}: P(מוביל) תאורטי={p[0]:.3f}  נמדד ב-1000 דגימות={sample.count(0)/1000:.3f}")


def simulate(T, runs=1000, n_easy=190, n_decision=10):
    pe, pd = softmax(z_easy, T), softmax(z_decision, T)
    same = 0
    for _ in range(runs):
        same += all(random.random() < pe[0] for _ in range(n_easy)) and \
                all(random.random() < pd[0] for _ in range(n_decision))
    return same / runs


print()
random.seed(1)
for T in [0.2, 0.7, 1.0]:
    print(f"T={T}: אנליטי={p_greedy_path(T, z_easy, z_decision):.4f}  סימולציה={simulate(T):.3f}")

<div dir="rtl">

## שאלות ניתוח — חלק ג'

**(א)** הנוסחה של `p_greedy_path` מניחה שהטוקנים **בלתי-תלויים**. האם ההנחה סבירה במודל שמייצר טוקן אחר טוקן, כשכל טוקן שנבחר הופך לחלק מהקלט של הבא? מה קורה בפועל אחרי שטוקן אחד "סוטה"?

**תשובה:** _(כתבו כאן)_

**(ב)** הוכיחו את הרמז: כתבו את השורה האלגברית שמראה שחיסור קבוע $c$ מכל הציונים $z_t$ לא משנה את $p(t)$. איזה $c$ בחרתם, ולמה הוא מונע גלישה?

**תשובה:** _(כתבו כאן)_

**(ג)** ב-$T = 1$, טוקן קל בוחר את המוביל ב-99.7% מהמקרים. למה התשובה כולה כמעט אף פעם לא זהה לתשובה המובילה? חשבו **בנפרד** את התרומה של 190 הטוקנים הקלים ($\pi_\text{easy}^{190}$) ושל 10 נקודות ההחלטה ($\pi_\text{dec}^{10}$). מי תורם יותר לסטייה?

**תשובה:** _(כתבו כאן)_

**(ד)** הרצתם פרומפט פעמיים וקיבלתם קוד שונה — האם אחת התשובות שגויה? ואם קיבלתם קוד **זהה** פעמיים — מה זה מרמז על ההתפלגות של המודל בשאלה הזו?

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# תא חישוב לשאלה (ג)

<div dir="rtl">

---

# חלק ד' — הזיות: לממש בדיקת קיום, ולגלות מה היא מפספסת (8 דק')

## הרקע

**הזיה** היא פלט שנשמע סביר אבל שגוי. בהרצאה ראינו חמישה סוגי הזיות בקוד:

| סוג | מה קורה | איך נראית השגיאה בדרך כלל |
|---|---|---|
| פונקציה/מתודה | שם שלא קיים | `AttributeError` |
| פרמטר | פרמטר שהפונקציה לא מקבלת | `TypeError` — או **שום שגיאה** |
| חבילה | ספרייה שלא קיימת | `ModuleNotFoundError` / שגיאת התקנה |
| גרסה | קיים, אבל בגרסה אחרת | תלוי |
| התנהגות | הקוד רץ, אבל לא עושה מה שנאמר | **אין שגיאה** |

## הקוד החשוד

הקוד הבא אמור להוריד מידע על הספרייה `requests` מאתר PyPI ולהדפיס אותו. יש בו **ארבע** הזיות. **אל תריצו אותו עדיין.**

<pre dir="ltr">
import requests

URL = "https://pypi.org/pypi/requests/json"
r = requests.fetch(URL)
data = r.to_json()
print(data.get_value("info"))
r2 = requests.get(URL, retries=3)
</pre>

**תחזית (לפני הכול):** לכל שורה חשודה — סוג ההזיה לפי הטבלה, ואיזו חריגה תיזרק.

**התחזית שלי:** _(כתבו כאן)_

## בדיקה סטטית — בלי להריץ את הקוד החשוד

הפונקציה המובנית `hasattr(obj, name)` מחזירה `True` אם לאובייקט `obj` יש תכונה (פונקציה, מתודה, משתנה) בשם `name`, ו-`False` אחרת — **בלי** להפעיל אותה. עליה תבנו כלי בדיקה.

</div>

In [ ]:
def audit(obj, names):
    """For each name, reports whether obj really has an attribute with that name.
    returns: dict name -> bool, e.g. audit(dict, ["get", "get_value"]) -> {"get": True, "get_value": False}
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — חלק ד', ואז הבדיקה הסטטית של הקוד החשוד
def check_d():
    assert audit(dict, ["get", "get_value"]) == {"get": True, "get_value": False}, "audit: הדוגמה מה-docstring"
    assert audit(list, []) == {}, "audit: רשימת שמות ריקה"
    return True

check_d() and print("✓ חלק ד' — המימוש עובר את הבדיקות\n")
print("requests:         ", audit(requests, ["get", "fetch"]))
print("requests.Response:", audit(requests.Response, ["json", "to_json"]))
print("dict:             ", audit(dict, ["get", "get_value"]))
print("החתימה של requests.get:", inspect.signature(requests.get))

<div dir="rtl">

## בדיקה בהרצה

התא הבא מריץ כל שורה חשודה **בנפרד**, בתוך `try/except`, ומדפיס את סוג החריגה ואת ההודעה. (`exec` מריץ מחרוזת כקוד פייתון.)

</div>

In [ ]:
URL = "https://pypi.org/pypi/requests/json"
for line in ['r = requests.fetch(URL)',
             'r = requests.get(URL, timeout=10)',
             'data = r.to_json()',
             'data = r.json()',
             'print(data.get_value("info"))',
             'r2 = requests.get(URL, retries=3)']:
    try:
        exec(line)
        print("OK  ", line)
    except Exception as e:
        print("ERR ", line, "->", f"{type(e).__name__}: {e}")

<div dir="rtl">

## תיקון

כתבו גרסה מתוקנת של הקוד החשוד, שמדפיסה את הגרסה האחרונה של `requests` (הנתון נמצא ב-`data["info"]["version"]`). בלי ניסיונות חוזרים — רק הורדה, פענוח JSON והדפסה.

</div>

In [ ]:
# TODO: הקוד המתוקן

<div dir="rtl">

## שאלות ניתוח — חלק ד'

**(א)** איזו הזיה ה-`audit` שלכם **לא** יכול לתפוס, ולמה? מה בחתימה של `requests.get` "מסתיר" אותה? (`**kwargs` בחתימה פירושו: "הפונקציה מקבלת כל פרמטר בעל שם, ומעבירה אותו הלאה".)

**תשובה:** _(כתבו כאן)_

**(ב)** `retries` זרק `TypeError` רק כי `requests` מעבירה את `**kwargs` הלאה לפונקציה שבודקת אותם. בתא שלמטה, כתבו פונקציה קצרה משלכם בשם `download(url, **kwargs)` ש**מתעלמת** מפרמטרים לא מוכרים, והראו שהקריאה `download(URL, retries=3)` "עובדת". לאיזה סוג הזיה הפכה ההזיה, ולמה היא מסוכנת יותר?

**תשובה:** _(כתבו כאן)_

**(ג)** השוו לתחזית שכתבתם בתחילת החלק. מה היה שגוי, ומה למדתם?

**תשובה:** _(כתבו כאן)_

</div>

In [ ]:
# TODO: הפונקציה download לשאלה (ב), והקריאה אליה

<div dir="rtl">

---

# חלק ה' — ניסוי: המלצות חבילות של מודל אמיתי (6 דק')

## הרקע

במחקר שהוצג בהרצאה, מודלים מסחריים המליצו בממוצע על לפחות 5.2% חבילות **שלא קיימות**, והרבה מהשמות הבדויים **חזרו** בהרצות חוזרות. תוקף יכול לרשום שם כזה כחבילה אמיתית עם קוד זדוני ולחכות שמישהו יתקין אותה (**slopsquatting**). בניסוי הזה תבדקו כלי AI אמיתי.

## מדד העקביות — דמיון ג'קארד

כדי למדוד עד כמה שתי רשימות המלצות דומות, מתייחסים אליהן כאל **קבוצות** (בלי חשיבות לסדר ולכפילויות) ומחשבים:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- $|A \cap B|$ — כמה איברים משותפים לשתי הקבוצות (**חיתוך**).
- $|A \cup B|$ — כמה איברים יש בסך הכול באחת מהן לפחות (**איחוד**).
- $J = 1$: קבוצות זהות. $J = 0$: אין אף איבר משותף. **מוסכמה:** שתי קבוצות ריקות — $J = 1$.

**דוגמה ידנית:** $A = \{\text{pandas}, \text{openpyxl}\}$, $B = \{\text{pandas}, \text{xlsxwriter}\}$: חיתוך = 1, איחוד = 3, ולכן $J = 1/3$.

**בפייתון:** `set(a)` הופך רשימה לקבוצה; `a & b` הוא חיתוך, `a | b` הוא איחוד, ו-`len` נותן את הגודל.

</div>

In [ ]:
def jaccard(a, b):
    """Jaccard similarity of two collections, treated as sets: |A ∩ B| / |A ∪ B|.
    1.0 = identical sets, 0.0 = nothing in common. Two empty collections -> 1.0.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה — חלק ה'
def check_e():
    assert jaccard(["pandas", "openpyxl"], ["pandas", "xlsxwriter"]) == 1 / 3, "jaccard: הדוגמה הידנית"
    assert jaccard([], []) == 1.0, "jaccard: שתי קבוצות ריקות — 1.0"
    assert jaccard(["a"], ["b"]) == 0.0, "jaccard: אין חיתוך — 0.0"
    assert jaccard(["a", "a", "b"], ["b", "a"]) == 1.0, "jaccard: כפילויות וסדר לא משנים"
    return True

check_e() and print("✓ חלק ה' — המימוש עובר את הבדיקות")

<div dir="rtl">

## כלי האימות (נתון)

הפונקציה הבאה **בודקת** חבילה ב-PyPI **בלי להתקין אותה**. היא מציגה גם את **תאריך ההעלאה הראשונה** של החבילה. **כלל:** אף פעם לא מתקינים חבילה רק כי מודל המליץ עליה — קודם בודקים.

</div>

In [ ]:
def check_pypi(name):
    """Looks a package up on PyPI WITHOUT installing it."""
    r = requests.get(f"https://pypi.org/pypi/{name}/json", timeout=10)
    if r.status_code == 404:
        print(f"{name:25} ✗ לא קיים ב-PyPI")
        return False
    info = r.json()
    uploads = [f["upload_time_iso_8601"][:10] for files in info["releases"].values() for f in files]
    first = min(uploads) if uploads else "?"
    print(f"{name:25} ✓ קיים | גרסה אחרונה {info['info']['version']:10} | הועלה לראשונה {first}")
    return True


for name in ["pandas", "openpyxl", "excel-fast-writer-pro"]:   # בדיקת שפיות
    check_pypi(name)

<div dir="rtl">

## הניסוי

הריצו בכלי AI חינמי את הפרומפט הבא **שלוש פעמים, כל פעם בשיחה חדשה** (כדי שההרצות יהיו בלתי-תלויות):

> כתוב קוד Python שקורא קובץ CSV, מסנן שורות שבהן העמודה price גדולה מ-100, ושומר את התוצאה לקובץ Excel. ציין אילו ספריות צריך להתקין.

לכל הרצה, רשמו בתא הבא את רשימת הספריות שהומלצו (בשמות שבהם מתקינים אותן, למשל `openpyxl`), והריצו אותו. **כלי ה-AI ששימש:** _(כתבו כאן)_

</div>

In [ ]:
# TODO: מלאו את הספריות שהומלצו בכל הרצה
run1 = []
run2 = []
run3 = []

runs = [run1, run2, run3]
exists = {}                                   # name -> True/False (each name is checked once)
for i, run in enumerate(runs, 1):
    print(f"--- הרצה {i} ---")
    for name in run:
        if name not in exists:
            exists[name] = check_pypi(name)
        else:
            print(f"{name:25} (נבדק כבר: {'✓ קיים' if exists[name] else '✗ לא קיים'})")

if exists:
    missing = sum(not ok for ok in exists.values())
    print(f"\nחבילות ייחודיות: {len(exists)}, מתוכן לא קיימות: {missing} ({missing / len(exists):.0%})")
    print(f"ג'קארד 1-2: {jaccard(run1, run2):.2f} | 1-3: {jaccard(run1, run3):.2f} | 2-3: {jaccard(run2, run3):.2f}")

<div dir="rtl">

## שאלות ניתוח — חלק ה'

**(א)** קשרו את ערכי ג'קארד שקיבלתם לחלק ג': מה ג'קארד נמוך (או גבוה) מלמד על ההתפלגות של המודל בשאלה הזו?

**תשובה:** _(כתבו כאן)_

**(ב)** האם אפשר להשוות את שיעור החבילות הלא-קיימות שמדדתם לנתון מההרצאה (5.2%)? מה הייתם צריכים כדי להשוות ברצינות?

**תשובה:** _(כתבו כאן)_

**(ג)** למה `check_pypi` מציג את **תאריך ההעלאה הראשונה**? מה המשמעות של חבילה שקיימת, אבל הועלתה לראשונה לפני שבועיים, ושמה זהה לשם שמודלים נוטים להמציא?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ו' — מסקנה (2 דק')

חמישה משפטים, אחד לכל חלק. כל משפט בנוי כך: **מה מימשתם ומדדתם ← איזו חוליה במנגנון זה מראה ← מה עושים בעקבות זה בעבודה עם AI.**

1. **טוקניזציה:** _(כתבו כאן)_
2. **Embeddings:** _(כתבו כאן)_
3. **טמפרטורה:** _(כתבו כאן)_
4. **הזיות:** _(כתבו כאן)_
5. **ניסוי החבילות:** _(כתבו כאן)_

---

# בונוס

**1. BPE על עברית.** הוסיפו לקורפוס עברית: `text_he = text + " " + " ".join(["ממוצע"] * 8 + ["ממוצעים"] * 3 + ["מספר"] * 6 + ["מספרים"] * 4)`, הריצו `train_bpe(text_he, 20)`, ופרקו את `ממוצע`, `ממוצעים` ו-`מספרים`. מה קרה עכשיו, לעומת התוצאה של `ממוצע` בחלק א'? מה זה אומר על הקשר בין כמות העברית בנתוני האימון לבין איכות הטוקניזציה?

**2. ניסוי בין כלים.** חזרו על חלק ה' בכלי AI שני, וחשבו ג'קארד **בין הכלים** (איחוד ההמלצות של כל כלי). האם שני כלים מסכימים יותר או פחות משלוש הרצות של אותו כלי? הסבירו.

</div>

In [ ]:
# תא לבונוס 1

In [ ]:
# תא לבונוס 2

<div dir="rtl">

---

# בדיקה מסכמת — הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime → Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓`.

</div>

In [ ]:
for name, check in [("א' — word_frequencies", check_a1),
                    ("א' — count_pairs", check_a2),
                    ("א' — merge_pair", check_a3),
                    ("א' — merge_corpus", check_a4),
                    ("ב' — cosine", check_b),
                    ("ג' — softmax, p_greedy_path", check_c),
                    ("ד' — audit", check_d),
                    ("ה' — jaccard", check_e)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")